# RAISE-Vision

In [ ]:
#INITIATE PROTOCOL
from opentrons import protocol_api
import socket
from opentrons.types import *
import opentrons.execute

protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

In [ ]:
#SET UP LAYOUT
well = protocol.load_labware("corning_24_wellplate_3.4ml_flat", 8) 
dryer_tips = protocol.load_labware("opentrons_96_tiprack_20ul", 10)
tips = protocol.load_labware("opentrons_96_tiprack_20ul", 11)
left = protocol.load_instrument("p20_single_gen2", "left", tip_racks=[tips])
right = protocol.load_instrument('p1000_single_gen2', 'right') 


drypad = protocol.load_labware("raise_vision_drypad_tiprack", 6)
stage = protocol.load_labware("raise_vision_stage", 1)
trash = protocol.load_labware("raise_vision_trash", 5)

In [ ]:
#ACTIVATE CAMERA FUNCTIONS

class Camera:
    def __init__(self, ip_address: str, port: int):
        self.client_socket = socket.create_connection((ip_address, port))
    
    def take_image(self, fname: str) -> None:
        msg: str = f"take-image,{fname}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
        
    def new_folder(self, dir_name: str) -> None:
        msg: str = f"new-folder,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def analyze_images(self, dir_name: str) -> None:
        msg: str = f"analyze-images,{dir_name}"
        self.client_socket.send(msg.encode())
        self.client_socket.recv(1024)
    
    def __enter__(self):
        return self
    
    def close(self):
        self.client_socket.close()
    
    def __exit__(self, exc_type, exc_val, traceback):
        self.client_socket.close()

In [ ]:
#FUNCTIONS FOR RAISE-Vision WORKFLOW

#INITIAL DRYING PROTOCOL
def drySides4p(rolledDryerLoc):
    #STAGE & DRYER INSERT LOCATION DEFINITION
    stg = stage.wells()[1]
    dryer = dryer_tips.wells()[rolledDryerLoc]

    #PICK UP DRYER INSERT FROM DEFINED LOCATION
    left.pick_up_tip(dryer)
    
    #DRY RIGHT SIDE OF LENS MOUNT
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 65)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=6.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=9, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)

    #DRY FRONT SIDE OF LENS MOUNT
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=-6.7, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=-10.5, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)

    #DRY LEFT SIDE OF LENS MOUNT
    stage.set_offset(x= -11.5, y=0, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-8.1, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-11.5, y=0, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
      
    #DRY BACK SIDE OF LENS MOUNT
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(40), force_direct = False, speed = 35)
    left.move_to(stg.top(22), force_direct = False, speed = 25)
    
    stage.set_offset(x=-1.2, y=8, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 2)
    protocol.delay(5)
    
    stage.set_offset(x=-1.2, y=11, z=3)
    left.move_to(stg.top(22), force_direct = False, speed = 15)
    left.move_to(stg.top(40), force_direct = False, speed = 25)
    
    
    #DROP DRYER INSERT INTO THE DESIGNATED TRASH
    throw = trash.wells()[5]
    left.drop_tip(throw)


#SESSILE DROP MEASUREMENT
def droplet(j, run_dir):
    #STAGE DEFINITION & CALIBRATION PARAMETERS
    stage.set_offset(x=-1.27, y=0, z=0.00)
    stg = stage.wells()[1]
    
    #DROPLET PLACEMENT LOCATIONS 
    locA = stg.top(53.75)                                #HEIGHT TO DISPENSE DROPLET FROM PIPETTE
    locB = stg.top(51.75)                                #HEIGHT TO APPLY DROPLET TO LENS
    
    #PICK UP TIP
    left.pick_up_tip()
    
    #TAKE PRE-LENS IMAGE
    fname = run_dir + f"pre_lens_{j}.jpg"
    cam.take_image(fname)

    #ASPIRATE 5 uL of LIQUID
    left.aspirate(5, well["D1"])                         

    #CREATE DROPLET ON PIPETTE TIP     
    left.move_to(locA, force_direct = False)
    left.dispense(5, locA, rate = 1)

    #APPLY DROPLET TO LENS 
    left.move_to(locB, force_direct = True, speed = 0.5)
    left.move_to(locA, force_direct = True, speed = 2)
    
    #TAKE DROPLET IMAGE   
    protocol.delay(5)
    fname = run_dir + f"Image_{j}.jpg"
    cam.take_image(fname)
    protocol.delay(1)
    
    #DROP TIP IN TRASH
    left.drop_tip() 

#DROPLET REMOVAL FROM LENS
def dryDropletTiprack(wellNum):
    #STAGE DEFINITION & CALIBRATION PARAMETERS & DRYPAD LOCATION DEFINITION
    stage.set_offset(x=-1.27, y=0, z=0.00)
    stg = stage.wells()[1]
    loc = drypad.wells()[wellNum]

    #PICK UP DRYPAD FROM DEFINED LOCATION
    left.pick_up_tip(loc)

    #DRY DROPLET
    left.move_to(stg.top(140), force_direct = False)
    left.move_to(stg.top(52), force_direct = False, speed = 40)
    left.move_to(stg.top(49.3), force_direct = False, speed = 2)
    protocol.delay(3)
    left.move_to(stg.top(130), force_direct = False)
    
    #RETURN DRYPAD TO RACK
    left.drop_tip(loc)

In [ ]:
#SET INTIIAL dryer_tips LOCATION TO 0 (ASSUMING NEW TIPRACK)
dryNum = 0

In [ ]:
#ENSURE CameraServer.py HAS BEEN RUN ON THE ANACONDA PROMPT FIRST
#ESTABLISH RAISE VISION OT-2 CAMERA CONNECTION
IP_ADDRESS = "192.168.0.111"

PORT = 1200
cam = Camera(IP_ADDRESS, PORT)

In [ ]:
#RUN AFTER THE LENS AND LENS MOUNT HAS BEEN PLACED ON THE STAGE
#RAISE-Vision WORKFLOW
run_dir = '../20260817-senofilcon-4p4m-RV-RT (1)/'          #CHANGE DIRECTORY NAME ACCORDING TO TEST
cam.new_folder(run_dir)

#INITIAL DRYING PROTOCOL
drySides4p(dryNum)
dryNum += 1                                                 #UPDATE DRYER INSERT LOCATION
if (dryNum >= 96):
    dryNum = 0

protocol.delay(49 + 120)                                    #2 MIN AIR EXPOSURE UNTIL DROPLET APPLICATION = 49s DELAY

#SESSILE DROP CYCLE
for j in range(10):
    droplet(j, run_dir)                                     #APPLY DROPLET TO LENS & TAKE IMAGE
    dryDropletTiprack(j)                                    #REMOVE DROPLET FROM LENS

#POST EXPERIMENT
cam.analyze_images(run_dir)
left.drop_tip()

In [ ]:
#CLOSE CAMERA AFTER ALL TRIALS ARE COMPLETE
cam.close()